<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/17_regression_crossval/17_2_MLR/17_2_4_2_MLR_Ames_Part2_Revised.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MLR Predicting Housing Prices in Ames Iowa: Part 2
## Forward and Backward Selection with Cross-Validation

**In this notebook we will:**
1.  Load the pre-cleaned, pre-split Ames dataset from Part 1's `ames_cleaning.py` module.
2.  Build a baseline model using forward feature selection and cross-validation.
3.  Diagnose problems with the model: negative coefficients, residual patterns, and multicollinearity.
4.  Test polynomial features to capture non-linear relationships.
5.  Assess multicollinearity with VIF and correlation analysis.

**Data Source:** http://jse.amstat.org/v19n3/decock/AmesHousing.txt

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Download the cleaning module from GitHub
import urllib.request
module_url = "https://raw.githubusercontent.com/bsheese/cs377/main/17_regression_crossval/17_2_MLR/ames_cleaning.py"
urllib.request.urlretrieve(module_url, "ames_cleaning.py")
from ames_cleaning import load_and_clean_ames

# Load and clean the Ames dataset
# The function returns pre-split, fully cleaned data ready for modeling
X_train, X_test, y_train, y_test = load_and_clean_ames()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")
print(f"\nTarget variable (y_train) is log-transformed SalePrice:")
print(y_train.describe())

### A Note on Data Cleaning

In Part 1, we walked through each data cleaning decision step-by-step: resolving meaningful NAs, correcting data types, removing outliers, imputing missing values with `SimpleImputer`, log-transforming SalePrice, engineering composite features, ordinal encoding quality variables, and one-hot encoding nominal features with `OneHotEncoder`.

That work has been consolidated into the `load_and_clean_ames()` function we called above. It returns `X_train, X_test, y_train, y_test` — fully model-ready data with:
- **No data leakage:** The train/test split happens before any statistics are computed.
- **Robust imputation:** `SimpleImputer` learns medians from train data only.
- **Proper encoding:** `OneHotEncoder(drop='first', handle_unknown='ignore')` prevents the dummy variable trap and gracefully handles unseen test categories.
- **Target separated:** `y_train` and `y_test` contain `log(SalePrice)`, so coefficients represent percentage effects.

If you want to see exactly what happens inside, open `ames_cleaning.py` — every step corresponds to a section of the Part 1 notebook.

## How Forward and Backward Selection Work

Instead of guessing which features matter, we let the data tell us. These algorithms search through possible feature combinations using cross-validation to evaluate each one.

### Forward Selection (Start Empty, Add Features)

1.  **Start with no features.** The model predicts the average price for every house.
2.  **Try adding each remaining feature, one at a time.** For each candidate, train the model with cross-validation and record the score.
3.  **Pick the feature that improves the score the most.** Add it to the model permanently.
4.  **Repeat.** Try adding each remaining feature to the growing set. Pick the best.
5.  **Stop when no feature improves the score.**

**Mini-example:** Imagine 3 features: A, B, C.
- Step 1: Try A alone (R²=0.50), B alone (R²=0.60), C alone (R²=0.40). Pick **B**.
- Step 2: Try B+A (R²=0.70), B+C (R²=0.65). Pick **A**.
- Step 3: Try B+A+C (R²=0.68). Adding C made things *worse*. Stop.
- Final model: **{A, B}**.

The `n_features_to_select='auto'` parameter means the algorithm keeps adding features as long as each new one improves the cross-validation score. When adding the next feature makes things worse, it stops.

### Backward Selection (Start Full, Remove Features)

The reverse approach:
1.  **Start with all features.**
2.  **Try removing each feature, one at a time.** See which removal hurts the score the *least*.
3.  **Remove the least important feature.**
4.  **Repeat** until removing any feature would significantly hurt performance.

Forward selection is faster when you have many features. Backward selection can catch interactions that forward selection misses, but it's computationally heavier.

**Note on Cross-Validation Folds:** We use only 2 CV folds to keep computation time manageable for this exercise (we have ~225 features!). In production, use 5-10 folds for more reliable estimates. Using fewer folds is a common trade-off when iterating quickly during model development.

In [ ]:
from typing import Dict, Any
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error


def perform_feature_selection_and_evaluation(
    X: pd.DataFrame,
    y: pd.Series,
    selection_direction: str = 'forward',
    cv_folds: int = 2
) -> Dict[str, Any]:
    """
    Performs feature selection and evaluates a Linear Regression model using cross-validation.
    Uses a Pipeline to prevent data leakage during cross-validation.

    Args:
        X (pd.DataFrame): Independent variables (training data).
        y (pd.Series): Dependent variable (training data).
        selection_direction (str): 'forward' or 'backward' for SequentialFeatureSelector.
        cv_folds (int): Number of cross-validation folds.

    Returns:
        dict: A dictionary containing the fitted pipeline, selected feature names, and CV metrics.
    """

    # 1. Build a Pipeline to prevent Data Leakage
    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('selector', SequentialFeatureSelector(
            LinearRegression(),
            n_features_to_select='auto',
            direction=selection_direction,
            cv=cv_folds
        )),
        ('model', LinearRegression())
    ])

    # 2. Perform Cross-Validation
    scoring_metrics = ['r2', 'neg_mean_squared_error']
    cv_results = cross_validate(pipeline, X, y, cv=cv_folds, scoring=scoring_metrics)

    cv_r2 = cv_results['test_r2']
    cv_mse = -cv_results['test_neg_mean_squared_error']

    print(f"Cross-validated R-squared (mean \u00b1 std): {cv_r2.mean():.4f} \u00b1 {cv_r2.std():.4f}")
    print(f"Cross-validated MSE (mean \u00b1 std): {cv_mse.mean():.6f} \u00b1 {cv_mse.std():.6f}\n")

    # 3. Fit the pipeline on ALL training data to get final features and coefficients
    pipeline.fit(X, y)

    final_selector = pipeline.named_steps['selector']
    final_model = pipeline.named_steps['model']

    selected_mask = final_selector.get_support()
    selected_feature_names = X.columns[selected_mask].tolist()

    # 4. Print results
    print(f"Intercept: {final_model.intercept_:.4f}\n")
    print(f"Standardized Coefficients (using {selection_direction} selection):")

    if selected_feature_names:
        max_feature_len = max(len(feature) for feature in selected_feature_names)
        for name, coef in zip(selected_feature_names, final_model.coef_):
            print(f"\t{name.ljust(max_feature_len)}:\t {coef:>7.4f}")
    else:
        print("\tNo features selected.")

    # 5. Return results
    return {
        'model_pipeline': pipeline,
        'selected_features': selected_feature_names,
        'cv_r2_mean': cv_r2.mean(),
        'cv_r2_std': cv_r2.std(),
        'cv_mse_mean': cv_mse.mean(),
        'cv_mse_std': cv_mse.std()
    }

### The Modeling Workflow

This function encapsulates the full pipeline: cross-validation with feature selection on training data, test set evaluation, and residual plotting. Since our data arrives pre-split from `load_and_clean_ames()`, we pass `X_train, X_test, y_train, y_test` directly.

In [ ]:
def run_modeling_workflow(
    X_train: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_test: pd.Series,
    selection_direction: str = 'forward',
    cv_folds: int = 2,
    plot_residuals: bool = True
) -> Dict[str, Any]:
    """
    Encapsulates the entire MLR modeling workflow: feature selection via CV on
    training data, then final evaluation on the held-out test set.

    Args:
        X_train, X_test: Feature matrices (pre-split).
        y_train, y_test: Target vectors (pre-split, log-transformed).
        selection_direction: 'forward' or 'backward'.
        cv_folds: Number of cross-validation folds.
        plot_residuals: Whether to plot residuals.

    Returns:
        dict with test metrics, selected features, fitted pipeline, residuals.
    """

    # 1. CROSS-VALIDATION & FEATURE SELECTION (on training data only)
    print("--- Running Cross-Validation on Training Data ---")
    results = perform_feature_selection_and_evaluation(
        X=X_train,
        y=y_train,
        selection_direction=selection_direction,
        cv_folds=cv_folds
    )

    final_model = results['model_pipeline']
    selected_features = results['selected_features']

    # 2. FINAL EVALUATION ON UNSEEN TEST DATA
    print("\n--- Running Final Evaluation on Unseen Test Data ---")
    test_predictions = final_model.predict(X_test)

    final_r2 = r2_score(y_test, test_predictions)
    final_mse = mean_squared_error(y_test, test_predictions)

    print(f"Final Test R-squared: {final_r2:.4f}")
    print(f"Final Test MSE:       {final_mse:.6f}")

    # 3. RESIDUAL ANALYSIS
    residuals = y_test - test_predictions

    if plot_residuals:
        plt.figure(figsize=(12, 5))

        plt.subplot(1, 2, 1)
        sns.histplot(residuals, kde=True)
        plt.title('Histogram of Residuals')
        plt.xlabel('Residual Value (log-dollars)')
        plt.ylabel('Frequency')

        plt.subplot(1, 2, 2)
        sns.scatterplot(x=test_predictions, y=residuals)
        plt.axhline(y=0, color='r', linestyle='--')
        plt.title('Residuals vs. Predicted Values')
        plt.xlabel('Predicted Values (log-dollars)')
        plt.ylabel('Residuals')

        plt.tight_layout()
        plt.show()

    return {
        'final_r2': final_r2,
        'final_mse': final_mse,
        'selected_features': selected_features,
        'model_pipeline': final_model,
        'residuals': residuals,
        'test_predictions': test_predictions
    }

### Forward Selection

Let's run forward selection on the full feature set (~225 features from our one-hot encoding). This may take a minute or two.

In [ ]:
forward_results = run_modeling_workflow(
    X_train, X_test, y_train, y_test,
    selection_direction='forward', cv_folds=2
)

### Backward Selection

Now let's run the same workflow with backward selection to see if it picks different features. Note: backward selection starts with ALL features and removes one at a time, so it will be significantly slower with ~225 features.

In [ ]:
backward_results = run_modeling_workflow(
    X_train, X_test, y_train, y_test,
    selection_direction='backward', cv_folds=2
)

## Interpreting Results

**Interpreting the Coefficients:**

Because our target is `log(SalePrice)`, each standardized coefficient represents the change in *log-price* per standard deviation change in the feature. In practical terms:
- A coefficient of 0.10 means a 1-standard-deviation increase in that feature raises the predicted price by approximately 10%.
- A negative coefficient means the feature *decreases* the predicted price.

**Why might Bedroom AbvGr be negative?**

Because `Total_Square_Footage` is already in the model, holding total area constant. If you have two 1,500 sq ft houses:
- House A has 3 bedrooms (500 sq ft each) — spacious rooms.
- House B has 5 bedrooms (300 sq ft each) — cramped rooms.

House B is worth *less* because the rooms are smaller. The negative coefficient captures this real-world tradeoff.

**What are the residuals telling us?**

Because we log-transformed SalePrice in Part 1, the residuals should be roughly evenly distributed (no funnel shape). If you still see a funnel, it suggests additional non-linearity or heteroscedasticity that the log transform didn't fully resolve.

**Multicollinearity:**

Many of our features are correlated with each other. We'll assess this formally with VIF later in the notebook.

### A Reminder: Why the Target is Log-Transformed

In Part 1, we applied `np.log(SalePrice)` before splitting features from target. This was done because:

1. **SalePrice is heavily right-skewed** — a few very expensive homes dominate the error function.
2. **Log compression stabilizes variance** — the model now pays equal attention to percentage errors across all price ranges.
3. **Coefficients become interpretable as percentage effects** — a coefficient of 0.05 means "a 1-SD increase in this feature raises price by ~5%."

This is why our MSE values are very small (in log-units) rather than in the billions (raw dollars).

### Add Polynomials to Model Non-Linear Features

In standard linear regression, the model assumes a strict, straight-line relationship between features and the target. However, data sometimes follows a curved pattern.

To capture curves, we add squared features ($X^2$) that allow the regression line to bend.

Key points:
1. **It is still "Linear" Regression:** The model is linear in its *parameters*. The squared term is just a new feature.
2. **Centering before squaring:** A base feature ($X$) and its square ($X^2$) are highly correlated. Standardizing before squaring reduces this "structural multicollinearity."
3. **Don't overfit:** Rarely go above degree 2 or 3.

**Example candidates for the Ames data:**
- `Overall Qual²` — luxury finishes command an exponential premium
- `Garage Cars²` — diminishing returns on garage capacity
- `Total_Square_Footage²` — diminishing returns on house size

In [ ]:
# Standardize selected features before squaring (to avoid structural multicollinearity)
from sklearn.preprocessing import StandardScaler

poly_base_features = ['Overall Qual', 'Garage Cars', 'Total_Square_Footage']

# Check which features actually exist in X_train
poly_base_features = [f for f in poly_base_features if f in X_train.columns]
print(f"Creating polynomial features for: {poly_base_features}")

scaler = StandardScaler()
train_scaled = scaler.fit_transform(X_train[poly_base_features])
test_scaled = scaler.transform(X_test[poly_base_features])

# Create squared terms
train_poly_df = pd.DataFrame(train_scaled, columns=poly_base_features, index=X_train.index)
test_poly_df = pd.DataFrame(test_scaled, columns=poly_base_features, index=X_test.index)

X_train_poly = X_train.copy()
X_test_poly = X_test.copy()

for col in poly_base_features:
    X_train_poly[f'{col}_Sqr'] = train_poly_df[col] ** 2
    X_test_poly[f'{col}_Sqr'] = test_poly_df[col] ** 2

# Run through the same workflow (forward selection)
poly_results = run_modeling_workflow(
    X_train_poly, X_test_poly, y_train, y_test,
    selection_direction='forward', cv_folds=2
)

### Did the Polynomials Help?

Examine whether forward selection chose any of the polynomial features (`_Sqr` suffix). If not, it means that given the features already selected, the polynomial terms didn't add enough *additional* explanatory power to justify their inclusion.

This doesn't mean non-linear relationships don't exist. Forward selection is conservative — it only adds features that clearly improve the cross-validation score.

In Part 3, we'll see that regularization (Ridge, Lasso) handles this more gracefully. Instead of an all-or-nothing decision, regularization can shrink polynomial coefficients toward zero while still allowing them to contribute.

---

### Assessing Multicollinearity with VIF

In regression analysis, your input features should ideally be strongly correlated with your target variable, but not with each other. When independent variables overlap too much (multicollinearity), it confuses the model, making individual coefficients unstable.

The Variance Inflation Factor (VIF) measures how much the variance of a coefficient is "inflated" due to correlation with other predictors.

**VIF Interpretation:**
- VIF = 1: No correlation with other features
- VIF 1-5: Moderate correlation (usually acceptable)
- VIF > 5: High correlation — consider removing or combining features
- VIF > 10: Very high correlation — likely problematic

**Note:** VIF is most meaningful for continuous/ordinal features. One-hot encoded dummy variables can show inflated VIF values due to their binary nature, so we'll focus on the numeric features.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

def calculate_vif(X: pd.DataFrame, threshold: float = 5.0) -> pd.DataFrame:
    """
    Calculate Variance Inflation Factor (VIF) for each feature.

    Args:
        X: DataFrame with numeric features (no NaN)
        threshold: VIF value above which features are flagged (default: 5.0)

    Returns:
        DataFrame with features and their VIF values, sorted descending
    """
    X_with_const = X.assign(const=1)

    vif_data = []
    for i, col in enumerate(X.columns):
        vif_value = variance_inflation_factor(X_with_const.values, i)
        vif_data.append({'Feature': col, 'VIF': vif_value})

    vif_df = pd.DataFrame(vif_data).sort_values('VIF', ascending=False)

    high_vif = vif_df[vif_df['VIF'] > threshold]
    print(f"Features with VIF > {threshold} (potential multicollinearity issues):")
    if len(high_vif) > 0:
        print(high_vif.to_string(index=False))
    else:
        print("  None - all features below threshold")

    return vif_df

# Run VIF on all numeric features from X_train
# (exclude one-hot dummies which are binary and inflate VIF artificially)
numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()

# Filter out binary (0/1) columns which are likely one-hot dummies
non_binary_cols = [col for col in numeric_features
                   if X_train[col].nunique() > 2]

print(f"Checking VIF for {len(non_binary_cols)} non-binary numeric features:\n")
vif_results = calculate_vif(X_train[non_binary_cols], threshold=5.0)
print("\nAll VIF values:")
print(vif_results.to_string(index=False))

Examine the VIF results above. Features with VIF > 5 are highly correlated with other predictors. This is expected for features like `Total_Square_Footage` which was engineered from component features (though we dropped those components in Part 1).

Now let's check whether the features *selected by forward selection* still have multicollinearity issues.

In [ ]:
# Get the features that forward selection chose
selected = forward_results['selected_features']

# Filter to only numeric selected features for correlation
selected_numeric = [f for f in selected if f in X_train.select_dtypes(include=[np.number]).columns]

if len(selected_numeric) > 2:
    corr_matrix = X_train[selected_numeric].corr()
    plt.figure(figsize=(10, 8))
    sns.heatmap(corr_matrix, vmin=-1, vmax=1, annot=True, fmt='.2f', cmap='coolwarm')
    plt.title('Correlation Matrix of Forward-Selected Numeric Features')
    plt.tight_layout()
    plt.show()
else:
    print("Not enough numeric features selected to compute correlations.")

In [ ]:
# Find feature pairs with absolute correlation > 0.7
if len(selected_numeric) > 2:
    stacked_corr = corr_matrix.stack()
    high_corr = stacked_corr[(abs(stacked_corr) > 0.7) & (stacked_corr != 1)]
    high_corr = high_corr.sort_values(ascending=False)

    print("Features with absolute correlation > 0.7:")
    if not high_corr.empty:
        printed_pairs = set()
        for (feature1, feature2), correlation in high_corr.items():
            if (feature2, feature1) not in printed_pairs:
                print(f"  {feature1} - {feature2}: {correlation:.4f}")
                printed_pairs.add((feature1, feature2))
    else:
        print("  No correlations above 0.7 found (excluding self-correlation).")

### VIF on Forward-Selected Features

Let's check whether the features chosen by forward selection still have multicollinearity issues:

In [ ]:
# Check VIF on only the forward-selected numeric features
selected_numeric_non_binary = [col for col in selected_numeric
                                if X_train[col].nunique() > 2]

if len(selected_numeric_non_binary) > 1:
    print(f"\nVIF for {len(selected_numeric_non_binary)} forward-selected non-binary features:\n")
    vif_reduced = calculate_vif(X_train[selected_numeric_non_binary], threshold=5.0)
    print("\nAll VIF values:")
    print(vif_reduced.to_string(index=False))
else:
    print("Not enough non-binary numeric features to compute VIF.")

## Summary

We've built a model using forward selection on a rich feature set (~225 features from proper one-hot encoding). Key takeaways:

1.  **The log-transformed target (from Part 1) ensures** that our model treats percentage errors equally across all price ranges. Coefficients represent the change in log-price per standard deviation change in the feature.
2.  **Forward selection effectively reduces dimensionality** — from ~225 features down to a manageable subset that explains most of the variance.
3.  **Multicollinearity is pervasive in the full feature set** but forward selection naturally filters out the most redundant features.
4.  **Polynomial features may or may not be selected** — forward selection's all-or-nothing approach means they need to clearly improve CV score to be included.
5.  **Coefficient instability remains a concern.** When features are correlated, the algorithm picks one and ignores the others. A different random split might select different features.

**In Part 3, we'll address these issues with regularization.** Ridge, Lasso, and ElasticNet penalize large coefficients, making the model less sensitive to which correlated features are included. They also handle polynomial features more gracefully by shrinking their coefficients rather than dropping them entirely.